# Gait 3D v2: rigid, rotation-invariant knee tracker

This notebook runs the v2 scripts step by step. The original `calibrate_capture.ipynb` is kept unchanged as v1.
See `CHANGELOG.md` for every issue found in v1 and how it was fixed.

**Close any other notebook that holds the cameras before running a cell.**

## 0. Self-test (no camera needed)
Runs the synthetic stereo-rig tests: rotation invariance, missing-depth recovery, ghost legs, calibration.

In [ ]:
!python3 -m pytest -q tests

## 1. Capture calibration pairs
**v1 issue:** blind 4 s timer, focus at infinity, only about 7 usable pairs.
**v2:** saves only sharp, moved boards seen by both cameras, at the same focus used for tracking (`--lens 3.0` is about 33 cm). Cover the whole image, especially the corners.
Press `q` in the window to stop early.

In [ ]:
%run scripts/01_capture_calibration.py --lens 3.0 --count 40

## 2. Calibrate
**Check the report:** the baseline must match a ruler measurement between the lens centres, and the epipolar error should be below 0.5 px.

In [ ]:
%run scripts/02_calibrate.py --square-mm 25 --balance 0.0
print(open('calibration_report.txt').read())

## 3. Depth diagnostic
**v1 issue:** the SGBM range saturated, so every joint read 24.0 or 25.9 cm. This shows how close each setting can measure and the live saturation %.
It should stay near 0% with your legs in front of the walker.

In [ ]:
%run scripts/03_check_depth.py --lens 3.0 --frames 300

## 4. (Optional) Faster YOLO with NCNN

In [ ]:
%run scripts/05_export_ncnn.py --imgsz 416

## 5. Track
Stand straight for 2 s (bone calibration), then walk. Keys: `q` quit, `c` recalibrate, `s` save profile.
Use `--model yolov8n-pose.pt` if you skipped step 4.

In [ ]:
%run scripts/04_run_tracker.py --lens 3.0 --model yolov8n-pose_ncnn_model --imgsz 416 --csv walk_01.csv

## 6. Plot the session

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv('walk_01.csv')
for side, g in df.groupby('side'):
    plt.plot(g.t - df.t.min(), g.flexion_filtered_deg, label=f'{side} knee flexion')
plt.xlabel('time (s)'); plt.ylabel('flexion (deg, 0 = straight)'); plt.legend(); plt.grid(); plt.show()
print(df.groupby('side')[['thigh_mm','shin_mm']].describe())  # must be constant